# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook guides users through loading and exploring the FAIR^2 dataset using the `mlcroissant` library, following the Croissant schema standard.

### Dataset Source
The dataset is described via a Croissant schema JSON-LD accessible at:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1m{metadata.name}\033[0m: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their details by @id
print("Available record sets in dataset:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '(unnamed)')}")
    print(f"  Description: {rs.get('description', '')}")
    # List fields inside this record set
    if 'field' in rs and isinstance(rs['field'], list):
        print("  Fields:")
        for f in rs['field']:
            if isinstance(f, dict):
                print(f"    - @id: {f['@id']}, name: {f.get('name', '')}")
            else:
                print(f"    - {f}")
    elif 'field' in rs:
        # Single field
        f = rs['field']
        if isinstance(f, dict):
            print(f"    - @id: {f['@id']}, name: {f.get('name', '')}")
        else:
            print(f"    - {f}")
    print()

## 3. Data Extraction
Extract data from each available record set into Pandas DataFrames. Use the `@id`s identified above.

In [ ]:
dataframes = {}
# Collect record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

if not record_set_ids:
    print("No record sets found in this dataset. Check the dataset metadata for available data resources or distributions.")
else:
    for record_set_id in record_set_ids:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(records)} records from record set {record_set_id}.')
    # Show columns of the first record set if available
    first_rs = record_set_ids[0]
    print(f"\nColumns in first record set ('{first_rs}'):")
    print(dataframes[first_rs].columns.tolist() if not dataframes[first_rs].empty else "[No data]")
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA such as filtering records, normalizing numeric fields, and grouping data by key attributes.

> **Note:** Substitute `<record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` with real values from Section 2.

In [ ]:
# Example: Suppose a record set and numeric field (use real @ids from above overview)
example_record_set_id = None
example_numeric_field = None
example_group_field = None

if record_set_ids:
    # Pick the first non-empty dataframe and try to guess a numeric field
    for rs_id in record_set_ids:
        df = dataframes[rs_id]
        if not df.empty:
            example_record_set_id = rs_id
            # Find a numeric column
            for col in df.columns:
                if pd.api.types.is_numeric_dtype(df[col]):
                    example_numeric_field = col
                    break
            # Find a likely categorical (non-numeric) for grouping
            for col in df.columns:
                if not pd.api.types.is_numeric_dtype(df[col]):
                    example_group_field = col
                    break
            break


if example_record_set_id and example_numeric_field:
    df = dataframes[example_record_set_id]
    # EDA: filter
    threshold = df[example_numeric_field].mean() if not df[example_numeric_field].isnull().all() else 0
    filtered_df = df[df[example_numeric_field] > threshold]
    print(f"Filtered records in '{example_record_set_id}' where '{example_numeric_field}' > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalize
    filtered_df = filtered_df.copy()
    if filtered_df[example_numeric_field].std() != 0:
        filtered_df[f"{example_numeric_field}_normalized"] = (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
        print(f"\nNormalized '{example_numeric_field}' for filtered records:")
        print(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())
    else:
        print(f"Cannot normalize '{example_numeric_field}' because std = 0")
    # Grouping
    if example_group_field and example_group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(example_group_field)[example_numeric_field].mean().reset_index()
        print(f"\nGrouped filtered data by '{example_group_field}':")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA. Please check dataset or record set structure.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization example (if EDA section found usable fields):
import matplotlib.pyplot as plt

if example_record_set_id and example_numeric_field:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8,4))
    df[example_numeric_field].hist(bins=30)
    plt.title(f"Distribution of '{example_numeric_field}' in record set '{example_record_set_id}'")
    plt.xlabel(example_numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if example_group_field and example_group_field in df.columns:
        df.boxplot(column=example_numeric_field, by=example_group_field, vert=False, figsize=(10,4))
        plt.title(f"'{example_numeric_field}' by '{example_group_field}'")
        plt.suptitle("")
        plt.xlabel(example_numeric_field)
        plt.ylabel(example_group_field)
        plt.show()
else:
    print("No numeric/grouping fields to visualize. Please check prior output.")

## 6. Conclusion
In this notebook, we've shown how to use the `mlcroissant` library to load and inspect a dataset described by a Croissant schema. We've listed available record sets, fields, and demonstrated basic exploratory data analysis and simple visualizations using actual field and record set `@id` values.

For your own exploratory workflows, consult the data dictionary and field metadata to select specific variables relevant to your analysis. The FAIR^2 dataset provides ordered logistic regression outputs and associated survey results on rangeland management adoption in Northern Kenya, and is suitable for research, policy, or demonstration purposes.